In [1]:
import torch
import torchvision
import torchaudio
import transformers
import datasets
import peft
import vllm

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("vllm:", vllm.__version__)

torch: 2.8.0+cu128
torchvision: 0.23.0+cu128
torchaudio: 2.8.0+cu128
transformers: 4.57.6
datasets: 3.6.0
peft: 0.19.1
vllm: 0.10.2


## 1. 테스트 데이터 전처리

In [11]:
import json
import re
import pandas as pd
from typing import List, Dict
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

In [3]:
# 1. 허깅페이스 허브에서 데이터셋 로드
dataset = load_dataset("iamjoon/ecommerce-function-calling-datasets-korean", split="train")

In [4]:
# 테스트 비율 설정
test_ratio = 0.2

# 전체 길이와 테스트 데이터 크기 계산
total_len = len(dataset)
test_size = int(total_len * test_ratio)

# 앞에서부터 테스트 데이터, 나머지는 학습 데이터
test_indices = list(range(test_size))
train_indices = list(range(test_size, total_len))

In [5]:
# OpenAI 포맷으로 변환 함수
def format_conversations(sample):
    return {
        "messages": [
            {"role": "system", "content": sample["system_prompt"]},
            *sample["messages"]
        ]
    }

# 분할 및 변환
train_dataset = [format_conversations(dataset[i]) for i in train_indices]
test_dataset = [format_conversations(dataset[i]) for i in test_indices]

# 리스트를 다시 HuggingFace Dataset 객체로 변환
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

# 결과 확인
print(f"\n전체 데이터 분할 결과: Train {len(train_dataset)}개, Test {len(test_dataset)}개")


전체 데이터 분할 결과: Train 311개, Test 77개


In [6]:
def remove_think_blocks(text):
    if text is None:
        return ""

    text = str(text)

    text = text.replace("<think>\n\n</think>\n\n", "")

    return text.strip()

In [7]:
def to_chatml(data):
    """
    data: messages 리스트이거나 {"messages": [...]} 형태의 dict
    반환값: ChatML 포맷의 문자열
    """
    # data가 dict이고 'messages' 키가 있으면 messages 리스트를 꺼내고,
    # 아니면 data 자체를 messages 리스트로 간주
    messages = data.get("messages") if isinstance(data, dict) and "messages" in data else data

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

    text = remove_think_blocks(text)

    return text

In [8]:
def extract_examples(chatml: str) -> List[Dict[str, str]]:
    """
    ChatML 문자열에서 각 assistant 응답을 분리하여
    'input'과 'label' 쌍을 생성합니다.
    'input'은 해당 assistant 응답 직전까지의 모든 대화 + '<|im_start|>assistant',
    'label'은 해당 assistant의 응답 내용입니다.
    """
    examples: List[Dict[str, str]] = []
    pattern = re.compile(r'<\|im_start\|>assistant(.*?)(?=<\|im_end\|>)', re.DOTALL)

    for match in pattern.finditer(chatml):
        start_idx = match.start()
        input_text = chatml[:start_idx].strip() + '\n<|im_start|>assistant'
        label_text = match.group(1).strip()

        input_text = remove_think_blocks(input_text)
        label_text = remove_think_blocks(label_text)

        examples.append({
            "input": input_text,
            "label": label_text
        })

    return examples

In [9]:
# Qwen3-4B 로컬 스냅샷 경로에서 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("/workspace/models/Qwen3-4B")

In [12]:
prompt_lst = []
label_lst = []

for item in test_dataset:
    chatml = to_chatml(item)  # ChatML 문자열로 변환
    examples = extract_examples(chatml)  # assistant 응답 단위로 분리

    for ex in examples:
        prompt_lst.append(ex['input'])
        label_lst.append(ex['label'])

In [13]:
print(prompt_lst[10])

<|im_start|>system
당신은 상준몰의 AI 상담사입니다. 성심성의껏 상담하십시오.

로그인한 사용자의 현재 ID: U006
오늘 날짜: 2024-02-02

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "add_to_cart", "description": "사용자의 장바구니에 지정된 상품(product_id)과 수량(quantity)을 추가합니다. 동일 상품이 이미 있으면 수량을 증가시키고, 새 항목으로 추가합니다.", "parameters": {"type": "object", "properties": {"user_id": {"type": "string", "description": "장바구니에 상품을 추가할 사용자의 고유 식별자 (예: 'U001')"}, "product_id": {"type": "string", "description": "장바구니에 추가할 상품의 고유 식별자 (예: 'P003')"}, "quantity": {"type": "integer", "description": "추가할 상품 수량 (기본값: 1)", "default": 1, "minimum": 1}}, "required": ["user_id", "product_id"], "additionalProperties": false}}}
{"type": "function", "function": {"name": "view_order_history", "description": "사용자의 전체 주문 내역을 반환합니다. 각 주문에 대해 주문 번호, 주문 일자, 총 결제 금액, 결제 상태, 배송 상태, 택배사, 운송장 번호, 배송 진행 단계, 주문에 포함된 상품명 목록을 제공

In [14]:
print(label_lst[10])

<tool_call>
{"name": "search_product", "arguments": {"keyword": "노트북"}}
</tool_call>


## 2. 모델 호출

In [15]:
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=2048,
    stop=["<|im_end|>"]
)

In [17]:
llm = LLM(model="/workspace/models/Qwen3-4B")

INFO 05-30 05:45:43 [utils.py:328] non-default args: {'disable_log_stats': True, 'model': '/workspace/models/Qwen3-4B'}
INFO 05-30 05:45:52 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 05-30 05:45:52 [__init__.py:1815] Using max model len 40960
INFO 05-30 05:45:54 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=7510) INFO 05-30 05:45:55 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=7510) INFO 05-30 05:45:55 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='/workspace/models/Qwen3-4B', speculative_config=None, tokenizer='/workspace/models/Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoni

[W530 05:45:58.475052005 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=7510) INFO 05-30 05:45:58 [gpu_model_runner.py:2338] Starting to load model /workspace/models/Qwen3-4B...
(EngineCore_DP0 pid=7510) INFO 05-30 05:45:59 [gpu_model_runner.py:2370] Loading model from scratch...
(EngineCore_DP0 pid=7510) INFO 05-30 05:45:59 [cuda.py:362] Using Flash Attention backend on V1 engine.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore_DP0 pid=7510) INFO 05-30 05:46:10 [default_loader.py:268] Loading weights took 10.94 seconds
(EngineCore_DP0 pid=7510) INFO 05-30 05:46:11 [gpu_model_runner.py:2392] Model loading took 7.5552 GiB and 11.271572 seconds
(EngineCore_DP0 pid=7510) INFO 05-30 05:46:17 [backends.py:539] Using cache directory: /root/.cache/vllm/torch_compile_cache/58b69f4896/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=7510) INFO 05-30 05:46:17 [backends.py:550] Dynamo bytecode transform time: 5.54 s
(EngineCore_DP0 pid=7510) INFO 05-30 05:46:22 [backends.py:194] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=7510) INFO 05-30 05:46:45 [backends.py:215] Compiling a graph for dynamic shape takes 27.63 s
(EngineCore_DP0 pid=7510) INFO 05-30 05:46:50 [monitor.py:34] torch.compile takes 33.17 s in total
(EngineCore_DP0 pid=7510) INFO 05-30 05:46:51 [gpu_worker.py:298] Available KV cache memory: 62.32 GiB
(EngineCore_DP0 pid=7510) INFO 05-30 05:46:52 [kv_cache_util

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:03<00:00, 21.27it/s]


(EngineCore_DP0 pid=7510) INFO 05-30 05:46:56 [gpu_model_runner.py:3118] Graph capturing finished in 4 secs, took 0.55 GiB
(EngineCore_DP0 pid=7510) INFO 05-30 05:46:56 [gpu_worker.py:391] Free memory on device (78.76/79.25 GiB) on startup. Desired GPU memory utilization is (0.9, 71.32 GiB). Actual usage is 7.56 GiB for weight, 1.43 GiB for peak activation, 0.02 GiB for non-torch memory, and 0.55 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=66174440243` to fit into requested memory, or `--kv-cache-memory=74158678016` to fully utilize gpu memory. Current kv cache memory in use is 66918929203 bytes.
(EngineCore_DP0 pid=7510) INFO 05-30 05:46:56 [core.py:218] init engine (profile, create kv cache, warmup model) took 44.88 seconds
INFO 05-30 05:46:57 [llm.py:295] Supported_tasks: ['generate']
INFO 05-30 05:46:57 [__init__.py:36] No IOProcessor plugins requested by the model


In [18]:
base_model_outputs = llm.generate(prompt_lst, sampling_params)

Adding requests:   0%|          | 0/655 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/655 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [19]:
base_model_text_results = [sample.outputs[0].text.strip() for sample in base_model_outputs]

In [20]:
print(base_model_text_results[10])

<think>
Okay, the user just asked for recommendations on popular notebooks. Let me see. They previously checked their order history, so now they're looking for product suggestions.

First, I need to figure out which function to use. The available functions include search_product, which allows searching by keyword and category. Since they mentioned "인기 있는 노트북," the keyword is "노트북" and category might be "전자기기" or something similar.

The search_product function requires a keyword and optionally a category. The user didn't specify a category, but "노트북" is a product type, so maybe the category is "전자기기" or "컴퓨터". But the function's parameters don't require category, just the keyword. However, the description says it can be filtered by category. 

Wait, the user is asking for popular notebooks. So the keyword is "노트북", and maybe the category is "전자기기" to narrow it down. But the function's parameters have "category" as optional. So I should call search_product with keyword "노트북" and maybe ca

## 3. 평가 결과 저장

In [21]:
df = pd.DataFrame({
    "prompt": prompt_lst,
    "label": label_lst,
    "output": base_model_text_results
})

# 저장
df.to_csv("base_model_evaluation_results.csv", index=False, encoding="utf-8-sig")

In [22]:
df

,prompt,label,output
0,<|im_start|>system\n당신은 상준몰의 AI 상담사입니다. 성심성의껏 ...,안녕하세요! 상준몰 AI 상담사입니다. 무엇을 도와드릴까요?,"<think>\nOkay, the user said ""안녕하세요."" which me..."
1,<|im_start|>system\n당신은 상준몰의 AI 상담사입니다. 성심성의껏 ...,"<tool_call>\n{""name"": ""view_user_profile"", ""ar...","<think>\nOkay, the user wants to check their a..."
2,<|im_start|>system\n당신은 상준몰의 AI 상담사입니다. 성심성의껏 ...,고객님의 주소는 '서울특별시 강남구 테헤란로 123'으로 등록되어 있습니다. 다른 ...,"<think>\nOkay, the user asked to check their a..."
3,<|im_start|>system\n당신은 상준몰의 AI 상담사입니다. 성심성의껏 ...,"<tool_call>\n{""name"": ""view_order_history"", ""a...","<think>\nOkay, the user wants to know the ship..."
4,<|im_start|>system\n당신은 상준몰의 AI 상담사입니다. 성심성의껏 ...,"고객님께서 7월 10일에 주문하신 '무선 이어폰'은 현재 배송중이며, 택배사는 한진...","<think>\nOkay, let's see. The user asked about..."
...,...,...,...
650,<|im_start|>system\n당신은 상준몰의 AI 상담사입니다. 성심성의껏 ...,"<tool_call>\n{""name"": ""search_policy_info"", ""a...","<think>\nOkay, the user just asked about the s..."
651,<|im_start|>system\n당신은 상준몰의 AI 상담사입니다. 성심성의껏 ...,"배송 정책에 따르면, 결제 완료 후 3일 이내에 출고되며, 배송비는 2,500원입니...","<think>\nOkay, the user asked about the shippi..."
652,<|im_start|>system\n당신은 상준몰의 AI 상담사입니다. 성심성의껏 ...,"죄송하지만, 날씨 정보는 제공해드릴 수 없습니다.","<think>\nOkay, the user asked about the weathe..."
653,<|im_start|>system\n당신은 상준몰의 AI 상담사입니다. 성심성의껏 ...,"네, 더 궁금하신 사항이 있으신가요?","<think>\nOkay, the user just said ""아, 그럼 됐어요.""..."


In [35]:
for label, pred in zip(df['label'].to_list()[:50], df['output'].to_list()[:50]):
    print('레이블 :', label)
    print('--' * 50)
    # <think></think>는 제외하고 출력
    # print('모델의 예측:', pred.split('</think>')[1].strip())
    print('모델의 예측:', pred)
    print('==' * 50)

레이블 : 안녕하세요! 상준몰 AI 상담사입니다. 무엇을 도와드릴까요?
----------------------------------------------------------------------------------------------------
모델의 예측: <think>
Okay, the user said "안녕하세요." which means "Hello" in Korean. I need to respond politely. Since they're greeting me, I should acknowledge their hello and offer assistance. Let me check the available functions to see if I need to call any immediately. The functions are for order details, cart, order history, product search, adding/removing from cart, policy info, and user profile. But the user hasn't asked for anything specific yet. So, no function calls are needed right now. Just a friendly response to start the conversation.
</think>

안녕하세요! 상준몰 AI 상담사입니다. 어떻게 도와드릴까요? 😊
레이블 : <tool_call>
{"name": "view_user_profile", "arguments": {"user_id": "U002"}}
</tool_call>
----------------------------------------------------------------------------------------------------
모델의 예측: <think>
Okay, the user wants to check their address. Let me see

## 4. 평가

아래 코드는 펑션 콜링 성능을 평가하기 위한 Python 함수를 구현한 것입니다. 요청하신 세 가지 메트릭을 다음과 같이 구현했습니다:

- tool_selection: 함수 이름의 일치 여부를 평가합니다.
- params_selection: 파라미터 키(예: user_id)의 일치 여부를 평가합니다.
- params_value_accuracy: 파라미터 값(예: "U002")의 일치 여부를 평가합니다.

In [29]:
def evaluate_function_calls(labels, predictions):
    """
    펑션 콜링 성능을 평가하는 함수
    
    Parameters:
    -----------
    labels : list
        정답 레이블 목록
    predictions : list
        모델이 예측한 결과 목록
        
    Returns:
    --------
    dict
        tool_selection: 함수 이름 일치율
        params_selection: 파라미터 키 일치율 
        params_value_accuracy: 파라미터 값 일치율
        total_samples: 전체 tool_call 샘플 수
    """
    # 결과 저장할 딕셔너리 초기화
    results = {
        'tool_selection': {'correct': 0, 'total': 0},
        'params_selection': {'correct': 0, 'total': 0},
        'params_value_accuracy': {'correct': 0, 'total': 0}
    }
    
    # tool_call 형식만 필터링하기 위한 정규표현식
    tool_call_pattern = re.compile(r'<tool_call>(.*?)</tool_call>', re.DOTALL)
    
    # 전체 샘플 중 tool_call 샘플 수
    tool_call_count = 0
    
    for label, pred in zip(labels, predictions):
        # tool_call 형식인지 확인
        label_match = tool_call_pattern.search(label)
        pred_match = tool_call_pattern.search(pred)
        
        # 레이블이 tool_call이 아니면 건너뛰기
        if not label_match:
            continue
        
        tool_call_count += 1
        
        # 예측이 tool_call 형식이 아니면 모든 지표가 틀린 것으로 처리
        if not pred_match:
            results['tool_selection']['total'] += 1
            results['params_selection']['total'] += 1
            results['params_value_accuracy']['total'] += 1
            continue
        
        # JSON 파싱
        try:
            label_json = json.loads(label_match.group(1))
            pred_json = json.loads(pred_match.group(1))
        except json.JSONDecodeError:
            # JSON 파싱 오류 시 모든 지표가 틀린 것으로 처리
            results['tool_selection']['total'] += 1
            results['params_selection']['total'] += 1
            results['params_value_accuracy']['total'] += 1
            continue
        
        # 1. 함수 이름 일치 여부 (tool_selection)
        results['tool_selection']['total'] += 1
        if label_json.get('name') == pred_json.get('name'):
            results['tool_selection']['correct'] += 1
        
        # 2. 파라미터 키 일치 여부 (params_selection)
        # 개별 파라미터별로 맞고 틀림을 채점
        label_params = set(label_json.get('arguments', {}).keys())
        pred_params = set(pred_json.get('arguments', {}).keys())
        
        # 각 파라미터마다 평가를 위해 모든 파라미터 순회
        for param in label_params:
            results['params_selection']['total'] += 1
            if param in pred_params:
                results['params_selection']['correct'] += 1
        
        # 예측에만 있는 추가 파라미터도 틀린 것으로 평가
        for param in pred_params:
            if param not in label_params:
                results['params_selection']['total'] += 1
                # correct는 증가 안 함 (틀린 것이므로)
        
        # 3. 파라미터 값 일치 여부 (params_value_accuracy)
        # 존재하는 공통 파라미터에 대해서만 값 일치 여부 평가
        label_args = label_json.get('arguments', {})
        pred_args = pred_json.get('arguments', {})
        
        # 공통으로 존재하는 파라미터 키 찾기
        common_params = label_params.intersection(pred_params)
        
        if common_params:  # 공통 파라미터가 있는 경우에만 평가
            results['params_value_accuracy']['total'] += 1
            
            # 공통 파라미터의 값이 모두 일치하는지 확인
            values_match = True
            for key in common_params:
                if label_args.get(key) != pred_args.get(key):
                    values_match = False
                    break
            
            if values_match:
                results['params_value_accuracy']['correct'] += 1
    
    # 최종 결과 계산
    final_results = {}
    for metric, counts in results.items():
        if counts['total'] > 0:
            final_results[metric] = counts['correct'] / counts['total']
        else:
            final_results[metric] = 0.0
    
    final_results['total_samples'] = tool_call_count
    
    return final_results

In [30]:
labels = [
    '안녕하세요! 상준몰 AI 상담사입니다. 무엇을 도와드릴까요?',
    '<tool_call>\n{"name": "view_user_profile", "arguments": {"user_id": "U002"}}\n</tool_call>',
    '고객님의 주소는 \'서울특별시 강남구 테헤란로 123\'으로 등록되어 있습니다. 다른 정보가 필요하신가요?',
    '<tool_call>\n{"name": "view_order_history", "arguments": {"user_id": "U002"}}\n</tool_call>'
]

predictions = [
    '안녕하세요! 상준몰 AI 상담사입니다. 무엇을 도와드릴까요?',
    '<tool_call>\n{"name": "view_user_profile", "arguments": {"user_id": "U002"}}\n</tool_call>',
    '고객님의 주소는 \'서울특별시 강남구 테헤란로 123\'로 등록되어 있습니다. 다른 정보가 필요하시면 말씀해 주세요.',
    '<tool_call>\n{"name": "view_order_history", "arguments": {"user_id": "U002"}}\n</tool_call>'
]

In [31]:
# 정상 케이스 평가
results = evaluate_function_calls(labels, predictions)
print("정상 케이스 평가 결과:")
for metric, value in results.items():
    if metric != 'total_samples':
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value}")

정상 케이스 평가 결과:
tool_selection: 100.00%
params_selection: 100.00%
params_value_accuracy: 100.00%
total_samples: 2


In [32]:
# 다른 예시 (에러 케이스)
labels_with_errors = [
    '<tool_call>\n{"name": "view_user_profile", "arguments": {"user_id": "U002"}}\n</tool_call>',
    '<tool_call>\n{"name": "search_product", "arguments": {"keyword": "노트북", "category": "전자기기"}}\n</tool_call>',
    '<tool_call>\n{"name": "check_stock", "arguments": {"product_id": "P001"}}\n</tool_call>'
]

predictions_with_errors = [
    '<tool_call>\n{"name": "view_profile", "arguments": {"user_id": "U002"}}\n</tool_call>',
    '<tool_call>\n{"name": "search_product", "arguments": {"keyword": "노트북"}}\n</tool_call>', 
    '죄송합니다. 재고 확인은 제품 번호가 필요합니다.'
]

In [33]:
print("\n에러 케이스 평가 결과:")
results_with_errors = evaluate_function_calls(labels_with_errors, predictions_with_errors)
for metric, value in results_with_errors.items():
    if metric != 'total_samples':
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value}")


에러 케이스 평가 결과:
tool_selection: 33.33%
params_selection: 50.00%
params_value_accuracy: 66.67%
total_samples: 3


**tool_selection: 33.33%**
- 세 개의 tool_call 중에서 하나만 함수 이름이 정확히 일치했습니다.
- 두 번째 예시 "search_product"가 일치했고, 나머지 두 개는 불일치했습니다.

**params_selection: 50.00%**
- 모든 파라미터를 개별적으로 평가합니다.
- 첫 번째 예시: label {"user_id"}, pred {"user_id"} → 1/1 맞음
- 두 번째 예시: label {"keyword", "category"}, pred {"keyword"} → 1/2 맞음 (keyword는 맞았고, category는 누락됨)
- 세 번째 예시: label {"product_id"}, pred 없음 → 0/1 맞음
- 총 4개 파라미터 중 2개 맞춤 → 50.00%

**params_value_accuracy: 66.67%**
- 공통 파라미터가 있는 두 개의 경우 중에서 두 개 모두 값이 일치했습니다.
- 첫 번째 예시에서 "user_id"의 값이 양쪽 모두 "U002"로 일치했습니다.
- 두 번째 예시에서 "keyword"의 값이 양쪽 모두 "노트북"으로 일치했습니다.
- 세 번째 예시는 tool_call 형식이 아니어서 평가되지 않았습니다.

In [34]:
labels = df['label'].to_list()
preds = df['output'].to_list()

results_with_errors = evaluate_function_calls(labels, preds)
for metric, value in results_with_errors.items():
    if metric != 'total_samples':
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value}")

tool_selection: 91.33%
params_selection: 86.85%
params_value_accuracy: 87.69%
total_samples: 196
